# Initialisation

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import colors, cm
from astropy.table import Table, join
from astropy.cosmology import Planck18
from astropy import units as u
from time import time

## Read data

In [ ]:
t0 = time()
#besta_dr1 = Table.read("data/BESTA/20260316/merged_catalogue.fits", unit_parse_strict='silent')
#besta_dr1 = Table.read("data/BESTA/stacked_catalogue.fits", unit_parse_strict='silent')
besta_dr1 = Table.read("data/BESTA/merged_ews_north.fits", unit_parse_strict='silent')
print(f'Done! (t={time()-t0:.2f} s)')

# Redshift bins

In [ ]:
zz = np.arange(0.01, 2.5, .01)
cosmic_time = Planck18.age(zz)
main_sequence = -np.log10(cosmic_time.to_value(u.yr))

In [ ]:
#redshift_bins = np.linspace(0.05, 1.95, 20)
redshift_bins = np.arange(0.05, 2.11, 0.1)
#redshift_bins = np.interp(np.arange(3, 13, .1)<<u.Gyr, cosmic_time[::-1], zz[::-1])[::-1]
redshift_centre = (redshift_bins[:-1] + redshift_bins[1:]) / 2
#redshift_colour = ['cyan', 'blue', 'green', 'orange', 'red']
redshift_labels = [f'z={z0:.2f}' for z0 in redshift_centre]
cosmic_time_centre_yr = Planck18.age(redshift_centre).to_value(u.yr)

In [ ]:
solid_angle_north = 0.15 / 4/np.pi
solid_angle_south = 0.43 / 4/np.pi
comoving_volume_Mpc3 = solid_angle_north * np.diff(Planck18.comoving_volume(redshift_bins).to_value(u.Mpc**3))

In [ ]:
plt.plot(zz, cosmic_time.to_value(u.Gyr), 'k-')
for edge in redshift_bins:
    plt.axvline(edge, c='k', ls='-', alpha=.2)
    plt.axhline(Planck18.age(edge).to_value(u.Gyr), c='k', ls='-', alpha=.2)
plt.ylabel("cosmic time [Gyr]")
plt.xlabel("redshift")


In [ ]:
redshift_bins

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(6, 4), squeeze=False)

ax = axes[0, 0]
ax.set_ylabel("BESTA mean $z$")
ax.set_xlabel("PHZ median $z$")
hh = ax.hist2d(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['Z__mean'], bins=redshift_bins, norm=colors.LogNorm(vmin=1), cmap="RdYlBu")
ax.plot([0., 1.9], [.1, 2.], 'k:')
ax.plot([.1, 2.], [.1, 2.], 'k--')
ax.plot([.1, 2.], [0., 1.9], 'k:')
plt.colorbar(hh[-1], ax=ax, label="number of galaxies per bin")

In [ ]:
'''
plt.hist2d(besta_dr1["Z__mean"], besta_dr1["Z__hi68"], bins=100, norm=colors.LogNorm())
plt.plot([0, 2], [0, 2], "k--")
plt.colorbar()

plt.hist2d(besta_dr1["Z__mean"], (besta_dr1["Z__lo68"]+besta_dr1["Z__hi68"])/2, bins=100, norm=colors.LogNorm())
plt.plot([0, 2], [0, 2], "k--")
plt.colorbar()

plt.hist2d(besta_dr1["Z__mean"], besta_dr1["Z__lo68"], bins=100, norm=colors.LogNorm())
plt.plot([0, 2], [0, 2], "k--")
plt.colorbar()

plt.hist2d(besta_dr1["Z__median"], (besta_dr1["Z__lo68"]+besta_dr1["Z__hi68"])/2, bins=100, norm=colors.LogNorm())
plt.plot([0, 2], [0, 2], "k--")
plt.colorbar()

plt.hist2d(besta_dr1["Z__std"], (besta_dr1["Z__hi68"]-besta_dr1["Z__lo68"])/2, bins=100, norm=colors.LogNorm())
plt.plot([0, 0.5], [0, 0.5], "k--")
plt.colorbar()

plt.hist2d(besta_dr1["Z__std"], np.fmax(besta_dr1["Z__hi68"]-besta_dr1["Z__median"], besta_dr1["Z__median"]-besta_dr1["Z__lo68"]), bins=100, norm=colors.LogNorm())
plt.plot([0, 0.5], [0, 0.5], "k--")
plt.colorbar()

hh, z_hi_edges, z_lo_edges = np.histogram2d(besta_dr1["Z__hi68"], besta_dr1["Z__lo68"], bins=100)

cumulative_lo = np.cumsum(hh, axis=0) # sum across z_lo
cumulative_lo /= cumulative_lo[-1, :][np.newaxis, :]
cumulative_hi = np.cumsum(hh, axis=1) # sum across z_hi
cumulative_hi /= cumulative_hi[:, -1][:, np.newaxis]

plt.pcolormesh(z_hi_edges, z_lo_edges, hh.T, norm=colors.LogNorm(), cmap="RdYlBu")
plt.colorbar()
plt.plot([0, 0.5], [0, 0.5], "k--")
plt.contour((z_hi_edges[:-1]+z_hi_edges[1:])/2, (z_lo_edges[:-1]+z_lo_edges[1:])/2, cumulative_hi.T, levels=[0.01, 0.1, 0.5], colors="k", linestyles=[":", "--", "-"])
plt.contour((z_hi_edges[:-1]+z_hi_edges[1:])/2, (z_lo_edges[:-1]+z_lo_edges[1:])/2, cumulative_lo.T, levels=[0.5, 0.9, 0.99], colors="g", linestyles=["-", "--", ":"])
plt.xlabel("z_hi")
plt.ylabel("z_lo");
''';

In [ ]:
plt.hist(besta_dr1["Z__lo68"] - besta_dr1["Z__hi68"], bins=100, cumulative=True, density=True)
plt.grid()
plt.xlabel("x_lo - x_hi")
plt.ylabel("frction");

In [ ]:
np.nanmedian(besta_dr1["Z__hi68"] - besta_dr1["Z__lo68"])

# Mass bins

In [ ]:
np.ma.median(besta_dr1["stellar_mass__hi68"] - besta_dr1["stellar_mass__lo68"])

In [ ]:
plt.hist(besta_dr1["stellar_mass__lo68"] - besta_dr1["stellar_mass__hi68"], bins=100, cumulative=True, density=True)
plt.grid()
plt.xlabel("M_lo - M_hi")
plt.ylabel("frction");

In [ ]:
delta_log_mass = 0.2
log_mass_bins = np.arange(6.5, 14.61, delta_log_mass)
log_mass_centre = (log_mass_bins[:-1] + log_mass_bins[1:]) / 2

In [ ]:
log_mass_threshold_z1 = 9.2
#log_mass_massive = 10.5
log_mass_threshold = log_mass_threshold_z1 + 2*np.log10(redshift_bins)
above_mass_threshold = log_mass_bins[:-1, np.newaxis] > log_mass_threshold[np.newaxis, 1:]

In [ ]:
counts, xedges, yedges = np.histogram2d(besta_dr1['Z__mean'], besta_dr1['stellar_mass__mean'], bins=(redshift_bins, log_mass_bins))
low_counts = counts < 100

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 6), width_ratios=(1, .05),
                         gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)

ax = axes[0, 0]
ax.set_ylabel('log ( M / M$_\odot$ )')
pcm = ax.pcolormesh(xedges, yedges, counts.T, norm=colors.LogNorm(vmin=1), cmap="RdYlBu")
plt.colorbar(pcm, cax=axes[0, -1], label="number of galaxies per bin")

'''
ax = axes[1, 0]
ax.set_ylabel('log ( M / M$_\odot$ )')
counts_per_volume = counts.T / comoving_volume_Mpc3[np.newaxis, :]
counts_per_volume[~ above_mass_threshold] = np.nan
counts_per_volume[low_counts.T] = np.nan
pcm = ax.pcolormesh(xedges, yedges, counts_per_volume, norm=colors.LogNorm(), cmap="nipy_spectral")
plt.colorbar(pcm, cax=axes[1, -1], label="galaxies per bin per comoving Mpc$^3$")
'''

for ax in axes[:, :-1].ravel():
    #for i, edge in enumerate(redshift_bins):
    #    ax.axvline(edge, c='k', ls=':')
    ax.set_xlim(redshift_bins[0]-.01, redshift_bins[-1]+.01)
    ax.set_ylim(log_mass_bins[0]-.05, log_mass_bins[-1]+.05)
    ax.grid(alpha=.2, c="k")

    #ax.axhline(10.25, c='k', ls=':')
    ax.plot(zz, 10.25 + 2*np.log10(Planck18.luminosity_distance(zz)/Planck18.luminosity_distance(redshift_bins[-1])), 'k--')
    #ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'k--')
    #ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'k--', lw=3)
    #ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'w--')
    #ax.axhline(log_mass_massive, c='w', lw=3)
    #ax.axhline(log_mass_massive, c='k', ls=':')
    #for i, edge in enumerate(redshift_bins):
    #    if i>0:
    #        bin_threshold = log_mass_threshold_z1 + 2*np.log10(edge)
    #        ax.plot([edge, redshift_bins[i-1]], [bin_threshold, bin_threshold], 'k:')

axes[-1, 0].set_xlabel("redshift");

# Multidimensional probability distributions

In [ ]:
log_ssfr9_bins = np.arange(-13.05, -7.99, .1)
log_ssfr9_centre = (log_ssfr9_bins[:-1] + log_ssfr9_bins[1:]) / 2

In [ ]:
t0 = time()
counts_ssfr9, edges = np.histogramdd((besta_dr1['Z__mean'].data, besta_dr1['stellar_mass__mean'].data, besta_dr1['log_ssfr_8p0__mean'].data),
                                     bins=(redshift_bins, log_mass_bins, log_ssfr9_bins))
#counts_ssfr9[low_counts] = np.nan
print(f'Done! (t={time()-t0:.2f} s)')

## Stellar mass

In [ ]:
mass_ssfr9 = counts_ssfr9 * 10**log_mass_centre[np.newaxis, :, np.newaxis]

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 15), width_ratios=(1, .05),
                         gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)

ax = axes[0, 0]
ax2 = ax.twinx()
idx = np.searchsorted(log_mass_bins, 10) - 1
#print(idx, log_mass_bins[idx], log_mass_centre[idx], log_mass_bins[idx+1])
line1, = ax.plot(redshift_centre, np.nansum(counts_ssfr9[:, idx:] / comoving_volume_Mpc3[:, np.newaxis, np.newaxis] , axis=(1, 2)), 'co:', label="number density per comoving volume")
line2, = ax2.plot(redshift_centre, np.nansum(mass_ssfr9[:, idx:] / comoving_volume_Mpc3[:, np.newaxis, np.newaxis] , axis=(1, 2)), 'rs:', label="mass density per comoving volume")
#line3 = ax2.axhline(Planck18.Ob0 * Planck18.critical_density(0).to_value(u.Msun/u.Mpc**3), ls="--", c="k", label=f'$\Omega_b={Planck18.Ob0:.2g}$')
ax.legend(handles=[line1, line2], title=f'$M>10^{{{log_mass_centre[idx]:g}}}$ M$_\odot$')
ax.set_ylabel(r"$\frac{dN}{dV}(M>10^{10}$ M$_\odot)$ [Mpc$^{-3}$]")
ax2.set_ylabel(r"$\frac{dM}{dV}(M>10^{10}$ M$_\odot)$ [M$_\odot$ Mpc$^{-3}$]")
#ax.set_yscale("log")
#ax2.set_yscale("log")
ax.set_ylim(0)
ax2.set_ylim(0)
#ax.plot(zz, cosmic_time.to_value(u.Gyr)*0.0003)
axes[0, -1].set_axis_off()

ax = axes[1, 0]
ax.set_ylabel('log ( M / M$_\odot$ )')
projected_counts = np.nansum(counts_ssfr9, axis=2).T
#projected_counts[~ above_mass_threshold] = np.nan
projected_counts /= comoving_volume_Mpc3[np.newaxis, :]
projected_counts /= delta_log_mass
pcm = ax.pcolormesh(redshift_bins, log_mass_bins, projected_counts, norm=colors.LogNorm(5e-9, 2e-2), cmap="RdYlBu")
plt.colorbar(pcm, cax=axes[1, -1], label=r"number density $\frac{dN}{dV\ d\,\log M}$ [Mpc$^{-3}$ dex$^{-1}$]")

ax = axes[2, 0]
ax.set_ylabel('log ( M / M$_\odot$ )')
projected_counts = np.nansum(mass_ssfr9, axis=2).T
#projected_counts[~ above_mass_threshold] = np.nan
projected_counts /= comoving_volume_Mpc3[np.newaxis, :]
projected_counts /= delta_log_mass
pcm = ax.pcolormesh(redshift_bins, log_mass_bins, projected_counts, norm=colors.LogNorm(5e4, 2e8), cmap="RdYlBu")
plt.colorbar(pcm, cax=axes[2, -1], label="mass per bin per comoving Mpc$^3$")
plt.colorbar(pcm, cax=axes[2, -1], label=r"mass density $\frac{dM}{dV\ d\,\log M}$ [M$_\odot$ Mpc$^{-3}$ dex$^{-1}$]")

for ax in axes[:, :-1].ravel():
    #for i, edge in enumerate(redshift_bins):
    #    ax.axvline(edge, c='k', ls=':')
    ax.set_xlim(redshift_bins[0]-.01, redshift_bins[-1]+.01)
    ax.grid(alpha=.2, c="k")

for ax in axes[1:, :-1].ravel():
    ax.set_ylim(6.75, 14.75)

    #ax.axhline(10.25, c='k', ls=':')
    ax.plot(zz, 10.25 + 2*np.log10(Planck18.luminosity_distance(zz)/Planck18.luminosity_distance(redshift_bins[-1])), 'k--')
    #ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'k--')
    #ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'k--', lw=3)
    #ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'w--')
    #ax.axhline(log_mass_massive, c='w', lw=3)
    #ax.axhline(log_mass_massive, c='k', ls=':')
    #for i, edge in enumerate(redshift_bins):
    #    if i>0:
    #        bin_threshold = log_mass_threshold_z1 + 2*np.log10(edge)
    #        ax.plot([edge, redshift_bins[i-1]], [bin_threshold, bin_threshold], 'k:')

axes[-1, 0].set_xlabel("redshift");

## Star Formation Rate

In [ ]:
SFR_ssfr9 = mass_ssfr9 * 10**log_ssfr9_centre[np.newaxis, np.newaxis, :]

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 15), width_ratios=(1, .05),
                         gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)

ax = axes[0, 0]
idx = np.searchsorted(log_mass_bins, 10) - 1
ax.plot(redshift_centre, np.nansum(SFR_ssfr9[:, idx:] / comoving_volume_Mpc3[:, np.newaxis, np.newaxis] , axis=(1, 2)), 'co:', label="SFR per comoving volume")
ax.legend(title=f'$M>10^{{{log_mass_centre[idx]:g}}}$ M$_\odot$')
ax.set_ylabel(r"$\frac{dM}{dt\ dV}(M>10^{10}$ M$_\odot)$ [M$_\odot$ Mpc$^{-3}$]")
axes[0, -1].set_axis_off()

ax = axes[1, 0]
ax.set_ylabel('log ( M / M$_\odot$ )')
projected_counts = np.nansum(SFR_ssfr9, axis=2).T
projected_counts /= comoving_volume_Mpc3[np.newaxis, :]
projected_counts /= delta_log_mass
pcm = ax.pcolormesh(redshift_bins, log_mass_bins, projected_counts, norm=colors.LogNorm(5e-6, 2e-2), cmap="RdYlBu")
plt.colorbar(pcm, cax=axes[1, -1], label="mass per bin per comoving Mpc$^3$")
plt.colorbar(pcm, cax=axes[1, -1], label=r"SFR density $\frac{dM}{dt\ dV\ d\,\log M}$ [M$_\odot$ yr$^{-1}$ Mpc$^{-3}$ dex$^{-1}$]")

ax = axes[2, 0]
ax.set_ylabel('log ( M / M$_\odot$ )')
projected_counts = np.nansum(SFR_ssfr9, axis=2).T
projected_counts /= np.nansum(mass_ssfr9, axis=2).T
projected_counts *= cosmic_time_centre_yr
pcm = ax.pcolormesh(redshift_bins, log_mass_bins, projected_counts, norm=colors.LogNorm(3e-2, 3), cmap="RdYlBu")
plt.colorbar(pcm, cax=axes[2, -1], label=r"average birthrate parameter $b = sSFR\ t$")

def plot_constant_b(log_m0, b):
    ax.plot(redshift_centre, log_m0 + b*np.log10(cosmic_time_centre_yr/cosmic_time_centre_yr[0]), 'k:')
    ax.text(redshift_centre[0], log_m0-.3, f'b = {b}')

plot_constant_b(12, .1)
plot_constant_b(11, .5)
plot_constant_b(10, 1)


for ax in axes[:, :-1].ravel():
    #for i, edge in enumerate(redshift_bins):
    #    ax.axvline(edge, c='k', ls=':')
    ax.set_xlim(redshift_bins[0]-.01, redshift_bins[-1]+.01)
    ax.grid(alpha=.2, c="k")

for ax in axes[1:, :-1].ravel():
    ax.set_ylim(6.75, 14.75)

    ax.plot(zz, 10.25 + 2*np.log10(Planck18.luminosity_distance(zz)/Planck18.luminosity_distance(redshift_bins[-1])), 'k--')
    #ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'k--')
    #ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'k--', lw=3)
    #ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'w--')
    #ax.axhline(log_mass_massive, c='w', lw=3)
    #ax.axhline(log_mass_massive, c='k', ls=':')
    #for i, edge in enumerate(redshift_bins):
    #    if i>0:
    #        bin_threshold = log_mass_threshold_z1 + 2*np.log10(edge)
    #        ax.plot([edge, redshift_bins[i-1]], [bin_threshold, bin_threshold], 'k:')

axes[-1, 0].set_xlabel("redshift");

In [ ]:
cosmic_time_centre_yr

# OLD

In [ ]:
raise -1

In [ ]:
## Mass function

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5), width_ratios=(1, .05),
                         gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)

norm = colors.LogNorm(vmin=1)
cmap = 'nipy_spectral'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

ax = axes[0, 0]
ax.set_ylabel('log ( M / M$_\odot$ )')
hh = ax.hist2d(besta_dr1['Z__mean'], besta_dr1['stellar_mass__mean'],
               bins=(redshift_bins, log_mass_bins), norm=norm, cmap=cmap)

plt.colorbar(hh[-1], cax=axes[0, -1], label="number of galaxies per bin")

'''

norm = colors.Normalize(vmin=8.75, vmax=11.25)
cmap = 'nipy_spectral'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

ax = axes[1, 0]
ax.set_ylabel("log ( sSFR8 / yr$^{-1}$ )")
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['log_ssfr_8p0__mean'], c=besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

#plt.colorbar(cm, cax=axes[1, -1], label="log ( sSFR9 / yr$^{-1}$ )")
plt.colorbar(cm, cax=axes[1, -1], label="log ( M / M$_\odot$ )")


ax = axes[2, 0]
ax.set_ylabel("log ( sSFR9 / yr$^{-1}$ )")
ax.set_ylim(-13.5, -8.5)
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['log_ssfr_9p0__mean'], c=besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

plt.colorbar(cm, cax=axes[2, -1], label="log ( M / M$_\odot$ )")
'''


for ax in axes[:, :-1].ravel():
    for i, edge in enumerate(redshift_bins):
        ax.axvline(edge, c='k', ls=':')
    #ax.set_xlim(-.01, redshift_bins[-1]+.01)

for ax in axes[0, :-1].ravel():
    ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'k-', lw=3)
    ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'w--')
    ax.axhline(log_mass_massive, c='w', lw=3)
    ax.axhline(log_mass_massive, c='k', ls=':')
    for i, edge in enumerate(redshift_bins):
        if i>0:
            bin_threshold = log_mass_threshold_z1 + 2*np.log10(edge)
            ax.plot([edge, redshift_bins[i-1]], [bin_threshold, bin_threshold], 'k:')

for ax in axes[1:, :-1].ravel():
    ax.plot(zz, main_sequence, 'k-', lw=3)
    ax.plot(zz, main_sequence, 'w--')

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(12, 8), width_ratios=(1, 1, 1, .05),
                         gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)

norm = colors.Normalize(vmin=0, vmax=3.75)
cmap = 'rainbow'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

ax = axes[0, 0]
ax.set_title('specz')
ax.set_ylabel('log ( M / M$_\odot$ )')
ax.set_ylim(8.5, 12.5)
#ax.scatter(specz['Z'], specz['stellar_mass__mean'], c=specz['bestfit_chi2'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[0, 1]
ax.set_title('photoz')
ax.yaxis.set_ticklabels('')
ax.set_ylim(8.5, 12.5)
#ax.scatter(photoz['PHZ_PP_MEDIAN_REDSHIFT'], photoz['stellar_mass__mean'], c=photoz['bestfit_chi2'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[0, 2]
ax.set_title('DR1')
ax.yaxis.set_ticklabels('')
ax.set_ylim(8.5, 12.5)
#ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['stellar_mass__mean'], c=besta_dr1['bestfit_chi2'], norm=norm, cmap=cmap, s=1, alpha=.25)
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

plt.colorbar(cm, cax=axes[0, -1], label="best-fit $\chi^2$")


norm = colors.Normalize(vmin=8.75, vmax=11.25)
cmap = 'nipy_spectral'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

ax = axes[1, 0]
ax.set_ylabel("log ( sSFR8 / yr$^{-1}$ )")
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('spectroscopic z')
#ax.scatter(specz['Z'], specz['log_ssfr_8p0__mean'], c=specz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[1, 1]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
#ax.scatter(photoz['PHZ_PP_MEDIAN_REDSHIFT'], photoz['log_ssfr_8p0__mean'], c=photoz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[1, 2]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['log_ssfr_8p0__mean'], c=besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

#plt.colorbar(cm, cax=axes[1, -1], label="log ( sSFR9 / yr$^{-1}$ )")
plt.colorbar(cm, cax=axes[1, -1], label="log ( M / M$_\odot$ )")


ax = axes[2, 0]
ax.set_ylabel("log ( sSFR9 / yr$^{-1}$ )")
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('spectroscopic z')
#ax.scatter(specz['Z'], specz['log_ssfr_9p0__mean'], c=specz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[2, 1]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
#ax.scatter(photoz['PHZ_PP_MEDIAN_REDSHIFT'], photoz['log_ssfr_9p0__mean'], c=photoz['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

ax = axes[2, 2]
ax.yaxis.set_ticklabels('')
ax.set_ylim(-13.5, -8.5)
ax.set_xlabel('PHZ median z')
ax.scatter(besta_dr1['PHZ_PP_MEDIAN_REDSHIFT'], besta_dr1['log_ssfr_9p0__mean'], c=besta_dr1['stellar_mass__mean'], norm=norm, cmap=cmap, s=1, alpha=.25)

plt.colorbar(cm, cax=axes[2, -1], label="log ( M / M$_\odot$ )")


for ax in axes[:, :-1].ravel():
    for i, edge in enumerate(redshift_bins):
        ax.axvline(edge, c='k', ls=':')
    ax.set_xlim(-.01, 1.01*redshift_bins[-1])

for ax in axes[0, :-1].ravel():
    ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'k-', lw=3)
    ax.plot(zz, log_mass_threshold_z1 + 2*np.log10(zz), 'w--')
    for i, edge in enumerate(redshift_bins):
        if i>0:
            bin_threshold = log_mass_threshold_z1 + 2*np.log10(edge)
            ax.plot([edge, redshift_bins[i-1]], [bin_threshold, bin_threshold], 'k:')

for ax in axes[1:, :-1].ravel():
    ax.plot(zz, main_sequence, 'k-', lw=3)
    ax.plot(zz, main_sequence, 'w--')

## Main sequence

In [ ]:
n_bins = redshift_centre.size
norm = colors.Normalize(vmin=-1., vmax=1.)
#cmap = 'nipy_spectral'
cmap = 'RdYlBu'
cm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

def plot_main_sequence(joint_table, redshift_column):
    fig, axes = plt.subplots(n_bins+1, 2, sharey=True, sharex=True, figsize=(12, n_bins*3), gridspec_kw={'wspace': 0, 'hspace': 0}, squeeze=False)
    for i, z0 in enumerate(redshift_centre):
        galaxies = np.where((joint_table[redshift_column] > redshift_bins[i]) & (joint_table[redshift_column] <= redshift_bins[i+1]))
        mass = joint_table['stellar_mass__mean'][galaxies]
        ssfr8 = joint_table['log_ssfr_8p0__mean'][galaxies]
        ssfr9 = joint_table['log_ssfr_9p0__mean'][galaxies]
        ax = axes[i, 0]
        sc = ax.scatter(mass, ssfr8, s=1, alpha=.5, c=ssfr8-ssfr9, norm=norm, cmap=cmap)
        ax.axvline(log_mass_threshold_z1+2*np.log10(redshift_bins[i+1]), c='k', ls=':')
        ax.axhline(np.interp(z0, zz, main_sequence), c='k', lw=3, label=f'z={z0:.2f}')
        ax.axhline(np.interp(z0, zz, main_sequence), c='w', ls='--')
        ax.grid(c='k', alpha=.2)
        ax = axes[i, 1]
        sc = ax.scatter(mass, ssfr9, s=1, alpha=.5, c=ssfr8-ssfr9, norm=norm, cmap=cmap)
        ax.axvline(log_mass_threshold_z1+2*np.log10(redshift_bins[i+1]), c='k', ls=':')
        ax.axhline(np.interp(z0, zz, main_sequence), c='k', lw=3, label=f'z={z0:.2f}')
        ax.axhline(np.interp(z0, zz, main_sequence), c='w', ls='--')
        ax.legend(loc='lower right')
        ax.grid(c='k', alpha=.2)
    ax.set_xlim(7.8, 12.2)
    ax.set_ylim(-13.5, -8.5)
    #axes[0, 0].set_ylabel("log( sSFR8 / yr )")
    #axes[1, 0].set_ylabel("log( sSFR9 / yr )")
    axes[0, 0].set_title(r"log( sSFR8 [yr$^{-1}$] )")
    axes[0, 1].set_title(r"log( sSFR9 [yr$^{-1}$] )")
    for ax in axes[-2, :]:
        ax.set_xlabel("log( M / M$_\odot$ )")
    for ax in axes[-1, :]:
        ax.set_axis_off()
    cb = plt.colorbar(cm, ax=axes[-1, :], label='log( sSFR8 / sSFR9 )', orientation="horizontal")
    plt.savefig("MS.png")
    #plt.close()

In [ ]:
#plot_main_sequence(specz, 'Z')
#plot_main_sequence(photoz, "PHZ_PP_MEDIAN_REDSHIFT")

#plot_main_sequence(besta_dr1, "PHZ_PP_MEDIAN_REDSHIFT")
t0 = time()
plot_main_sequence(besta_dr1, "Z__mean")
print(f'{time()-t0:.2f}')